# CUHK-X Small Model Track — Step 1: build the compact dataset

Runs entirely on Kaggle, so your own connection never carries the 44 GB.
Downloads the source from HuggingFace at Kaggle's bandwidth, extracts **one
modality at a time** (peak disk stays under the ~73 GB limit), packs
uniformly-sampled JPEG frames into blob shards plus a 3D-pose array, and
writes a few GB to `/kaggle/working`.

**Before running:**
1. Accept the terms at
   https://huggingface.co/datasets/Kevin-Pal/CUHK-X_Small_Model_Track
2. Add-ons → Secrets → add `HF_TOKEN` (a HuggingFace **read** token) and
   attach it to this notebook.
3. Settings → Accelerator **None** (this step is I/O bound — save GPU quota),
   Internet **On**, Persistence **Files**.

Then **Save Version → Save & Run All (Commit)** and let it run unattended.

In [ ]:
import os, sys, subprocess, shutil, time, json, glob

T0 = time.time()

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")

def disk():
    t, u, f = shutil.disk_usage("/kaggle")
    return f"disk: {u/1e9:.1f} GB used / {f/1e9:.1f} GB free"

def tsize(p):
    return sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(p) for f in fs) / 1e9

print(disk())

## Config

Measured from the real test set: Depth_Color and IR are 640x480 and present
for 100% of clips; Thermal is 320x240 at 97.5%; Skeleton is 3D pose present
for 100%; **Radar is empty for 207/405 clips so it is excluded**. Clips are
short — median 20 depth frames — so 16 sampled frames covers most of them.

In [ ]:
MODALITIES  = ["Skeleton", "Depth_Color", "IR", "Thermal"]
FRAMES      = 16     # image frames per clip
SKEL_FRAMES = 32     # pose is cheap, so sample it denser
SIZE        = 160    # stored; training random-crops to 144
QUALITY     = 90
CROP        = True   # foreground person crop

WORK    = "/kaggle/working"
SCRATCH = "/kaggle/temp"          # not persisted — holds the 44 GB of archives
RAW     = f"{SCRATCH}/raw"
EXTRACT = f"{SCRATCH}/extracted"
OUT     = f"{WORK}/compact"
for d in (RAW, EXTRACT, OUT):
    os.makedirs(d, exist_ok=True)

sh("pip install -q huggingface_hub hf_transfer pyarrow 2>&1 | tail -2", check=False)
sh("apt-get -qq install -y p7zip-full 2>&1 | tail -2", check=False)

## Preprocessing module

Identical code to the local `kaggle/prep.py`, so a local run and this notebook
produce the same shards.

In [ ]:
%%writefile prep.py
"""
Dataset compaction — single source of truth for local and Kaggle runs.

Two output families:

  image modalities (Depth_Color, IR, Thermal)
      <out>/<mod>/blob_NNN.bin      concatenated JPEG bytes
      <out>/<mod>/index.parquet     clip_id, split, user, trial, action_id,
                                    blob, offsets, lengths, t, size

  Skeleton
      <out>/Skeleton/poses.npy      float16 [N, T, 17, 3]
      <out>/Skeleton/index.parquet  clip_id, split, user, trial, action_id, row

Measured facts this is built around (from the real test set, 405 clips):
  * Depth_Color / IR / Skeleton are present for 100% of clips and are frame-
    aligned 1:1. Thermal covers 97.5% at ~2.2x the frame rate. Radar is empty
    for 207/405 clips, so it is not worth modelling.
  * Skeleton is Human3.6M 17-joint 3D, root-centred in x/y with z as height.
  * Clips are short: median 20 depth frames, min 2.
  * The archives carry macOS cruft (__MACOSX/, .DS_Store) and a stray .claude/
    directory inside the test root — all must be filtered out or they are
    silently ingested as clips.
"""
from __future__ import annotations

import json
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd

cv2.setNumThreads(1)

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
SHARD_CLIPS = 512
IMAGE_MODALITIES = ("Depth_Color", "IR", "Thermal")
SKELETON = "Skeleton"
N_JOINTS = 17

# Human3.6M 17-joint bone list, inferred from the measured joint heights
# (0 root at x=y=0, 3/6 feet at z~0, 10 head at z~1.23).
H36M_EDGES = [
    (0, 1), (1, 2), (2, 3),            # right leg
    (0, 4), (4, 5), (5, 6),            # left leg
    (0, 7), (7, 8), (8, 9), (9, 10),   # spine -> head
    (8, 11), (11, 12), (12, 13),       # left arm
    (8, 14), (14, 15), (15, 16),       # right arm
]


def is_junk(name):
    return name in ("__MACOSX", ".DS_Store", ".claude") or name.startswith("._")


def list_frames(d):
    try:
        names = [f for f in os.listdir(d)
                 if os.path.splitext(f)[1].lower() in IMG_EXT and not is_junk(f)]
    except OSError:
        return []

    def key(n):
        dig = "".join(c if c.isdigit() else " " for c in n).split()
        return ([int(x) for x in dig], n)

    return [os.path.join(d, f) for f in sorted(names, key=key)]


def sample_idx(n, t):
    if n <= 0:
        return []
    if n <= t:
        return list(range(n)) + [n - 1] * (t - n)
    return np.linspace(0, n - 1, t).round().astype(int).tolist()


# ------------------------------------------------------------------- images

def load_img(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    if img.dtype == np.uint16:
        lo, hi = np.percentile(img, [1, 99])
        img = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)
        img = (img * 255).astype(np.uint8)
    elif img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img


def foreground_box(img, modality):
    """Cheap deterministic person localisation; None means 'keep full frame'."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    g = cv2.GaussianBlur(g, (5, 5), 0)
    if modality.lower().startswith("thermal"):
        mask = (g >= np.percentile(g, 92)).astype(np.uint8) * 255
    else:
        _, mask = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 0.01 * g.size:
        return None
    x, y, w, h = cv2.boundingRect(c)
    return x, y, x + w, y + h


def union_box(boxes, shape, pad=0.15):
    boxes = [b for b in boxes if b]
    if not boxes:
        return None
    H, W = shape
    x0 = min(b[0] for b in boxes); y0 = min(b[1] for b in boxes)
    x1 = max(b[2] for b in boxes); y1 = max(b[3] for b in boxes)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(max(x1 - x0, y1 - y0) * (1 + 2 * pad), 0.30 * min(H, W))
    x0 = int(max(0, cx - side / 2)); x1 = int(min(W, cx + side / 2))
    y0 = int(max(0, cy - side / 2)); y1 = int(min(H, cy + side / 2))
    return None if (x1 - x0 < 16 or y1 - y0 < 16) else (x0, y0, x1, y1)


def encode_clip(a):
    clip_dir, modality, T, size, crop, quality = a
    frames = list_frames(clip_dir)
    if not frames:
        return None

    # A few recordings contain corrupt PNGs (observed on 4 test clips from the
    # 2025-06-01_12-11 session). Dropping the whole clip over one bad frame
    # would forfeit a test prediction outright, so substitute a neighbour and
    # only give up if nothing in the clip decodes.
    imgs, last = [], None
    for i in sample_idx(len(frames), T):
        im = load_img(frames[i])
        if im is None:
            im = last
        else:
            last = im
        imgs.append(im)
    if all(im is None for im in imgs):
        return None
    if imgs[0] is None:                       # backfill any leading failures
        first = next(im for im in imgs if im is not None)
        imgs = [first if im is None else im for im in imgs]
    # shapes can differ if a substitute came from a differently-sized frame
    h, w = imgs[0].shape[:2]
    imgs = [im if im.shape[:2] == (h, w) else cv2.resize(im, (w, h))
            for im in imgs]
    if crop:
        probe = imgs[:: max(1, len(imgs) // 4)][:4]
        box = union_box([foreground_box(im, modality) for im in probe],
                        imgs[0].shape[:2])
        if box:
            x0, y0, x1, y1 = box
            imgs = [im[y0:y1, x0:x1] for im in imgs]
    enc = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    out = []
    for im in imgs:
        im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
        ok, buf = cv2.imencode(".jpg", im, enc)
        if not ok:
            return None
        out.append(buf.tobytes())
    return clip_dir, len(frames), out


# ----------------------------------------------------------------- skeleton

def read_pose_json(path):
    """
    One frame -> [17,3] float32, or None.

    The file holds a list of detected people; keep the one with the best mean
    keypoint score (ties broken by bounding-box extent) so multi-person frames
    do not inject a bystander's pose.
    """
    try:
        with open(path, "r") as f:
            people = json.load(f)
    except Exception:
        return None
    if not isinstance(people, list) or not people:
        return None

    best, best_score = None, -1.0
    for p in people:
        kp = p.get("keypoints")
        if not kp or len(kp) < N_JOINTS:
            continue
        a = np.asarray(kp, dtype=np.float32)[:N_JOINTS]
        if a.shape[1] < 3:
            a = np.concatenate([a, np.zeros((N_JOINTS, 3 - a.shape[1]), np.float32)], 1)
        sc = p.get("keypoint_scores") or []
        score = float(np.mean(sc)) if len(sc) else 0.0
        score += 0.01 * float(np.ptp(a[:, 2]))     # prefer the fuller skeleton
        if score > best_score:
            best, best_score = a[:, :3], score
    return best


def encode_skeleton(a):
    clip_dir, T = a
    pdir = os.path.join(clip_dir, "predictions")
    root = pdir if os.path.isdir(pdir) else clip_dir
    try:
        files = sorted(f for f in os.listdir(root)
                       if f.endswith(".json") and not is_junk(f))
    except OSError:
        return None
    if not files:
        return None

    poses, last = [], None
    for i in sample_idx(len(files), T):
        p = read_pose_json(os.path.join(root, files[i]))
        if p is None:
            p = last if last is not None else np.zeros((N_JOINTS, 3), np.float32)
        last = p
        poses.append(p)
    return clip_dir, len(files), np.stack(poses).astype(np.float32)


def normalize_pose(seq):
    """
    Make the sequence subject- and placement-invariant.

    Centre on the root joint every frame (removes translation), then divide by
    the subject's own torso length (removes body-size differences between
    people, which is precisely the cross-subject nuisance). Height information
    survives as a ratio rather than absolute metres.
    """
    seq = seq.astype(np.float32).copy()
    root = seq[:, 0:1, :]
    seq = seq - root
    torso = np.linalg.norm(seq[:, 8, :] - seq[:, 0, :], axis=-1)   # root->thorax
    scale = np.median(torso[torso > 1e-3]) if np.any(torso > 1e-3) else 1.0
    return seq / max(float(scale), 1e-3)


# ---------------------------------------------------------------- discovery

def discover_train(train_root, modality):
    """HAR/data/<modality>/<action_id>_<name>/<user>/<trial>/"""
    mroot = os.path.join(train_root, modality)
    if not os.path.isdir(mroot):
        return []
    recs = []
    for action in sorted(os.listdir(mroot)):
        if is_junk(action):
            continue
        aroot = os.path.join(mroot, action)
        if not os.path.isdir(aroot):
            continue
        head = action.split("_")[0]
        if not head.isdigit():
            continue
        aid = int(head)
        for user in sorted(os.listdir(aroot)):
            if is_junk(user):
                continue
            uroot = os.path.join(aroot, user)
            if not os.path.isdir(uroot):
                continue
            for trial in sorted(os.listdir(uroot)):
                if is_junk(trial):
                    continue
                troot = os.path.join(uroot, trial)
                if os.path.isdir(troot):
                    recs.append(dict(clip_dir=troot, split="train", user=user,
                                     trial=trial, action_id=aid,
                                     clip_id=f"{action}/{user}/{trial}"))
    return recs


def discover_test(test_root, modality):
    """small_model_track_test/SM_test_XXXX/<modality>/"""
    if not os.path.isdir(test_root):
        return []
    recs = []
    for clip in sorted(os.listdir(test_root)):
        if not clip.startswith("SM_test"):      # skips .claude, .DS_Store, ...
            continue
        mdir = os.path.join(test_root, clip, modality)
        if os.path.isdir(mdir):
            recs.append(dict(clip_dir=mdir, split="test", user="TEST",
                             trial=clip, action_id=-1, clip_id=clip))
    return recs


def find_roots(extracted):
    """Locate HAR/data and the test clip root, ignoring the __MACOSX shadow tree."""
    train_root = test_root = None
    for dirpath, dirnames, _ in os.walk(extracted):
        dirnames[:] = [d for d in dirnames if not is_junk(d)]
        if "__MACOSX" in dirpath:
            continue
        base = os.path.basename(dirpath)
        if base == "data" and os.path.basename(os.path.dirname(dirpath)) == "HAR":
            train_root = train_root or dirpath
        if base == "small_model_track_test":
            test_root = test_root or dirpath
    return train_root, test_root


# ------------------------------------------------------------------ packing

def _progress(done, total, t0, every=1000):
    if done % every == 0:
        r = done / max(time.time() - t0, 1e-9)
        print(f"    {done}/{total}  {r:.0f} clips/s  "
              f"eta {(total-done)/max(r,1e-9)/60:.1f} min", flush=True)


def pack_images(modality, recs, out_root, T, size, crop, quality, workers):
    mout = os.path.join(out_root, modality)
    os.makedirs(mout, exist_ok=True)
    by_dir = {r["clip_dir"]: r for r in recs}
    jobs = [(r["clip_dir"], modality, T, size, crop, quality) for r in recs]

    rows, bi, off, done = [], 0, 0, 0
    t0 = time.time()
    bf = open(os.path.join(mout, f"blob_{bi:03d}.bin"), "wb")
    try:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            for fut in as_completed([ex.submit(encode_clip, j) for j in jobs]):
                res = fut.result(); done += 1
                _progress(done, len(jobs), t0)
                if res is None:
                    continue
                cd, nsrc, bufs = res
                r = by_dir[cd]
                if len(rows) and len(rows) % SHARD_CLIPS == 0:
                    bf.close(); bi += 1; off = 0
                    bf = open(os.path.join(mout, f"blob_{bi:03d}.bin"), "wb")
                offs, lens = [], []
                for b in bufs:
                    offs.append(off); lens.append(len(b)); bf.write(b); off += len(b)
                rows.append(dict(clip_id=r["clip_id"], split=r["split"],
                                 user=r["user"], trial=r["trial"],
                                 action_id=r["action_id"], n_src_frames=nsrc,
                                 blob=bi, offsets=offs, lengths=lens,
                                 t=len(bufs), size=size))
    finally:
        bf.close()

    df = pd.DataFrame(rows)
    df.to_parquet(os.path.join(mout, "index.parquet"), index=False)
    gb = sum(os.path.getsize(os.path.join(mout, f)) for f in os.listdir(mout)
             if f.endswith(".bin")) / 1e9
    print(f"  [{modality}] {len(df)} clips, {bi+1} blobs, {gb:.2f} GB, "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
    return df


def pack_skeleton(recs, out_root, T, workers):
    mout = os.path.join(out_root, SKELETON)
    os.makedirs(mout, exist_ok=True)
    by_dir = {r["clip_dir"]: r for r in recs}
    jobs = [(r["clip_dir"], T) for r in recs]

    rows, arrs, done = [], [], 0
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(encode_skeleton, j) for j in jobs]):
            res = fut.result(); done += 1
            _progress(done, len(jobs), t0, every=2000)
            if res is None:
                continue
            cd, nsrc, seq = res
            r = by_dir[cd]
            rows.append(dict(clip_id=r["clip_id"], split=r["split"],
                             user=r["user"], trial=r["trial"],
                             action_id=r["action_id"], n_src_frames=nsrc,
                             row=len(arrs), t=T))
            arrs.append(normalize_pose(seq).astype(np.float16))

    if not arrs:
        print(f"  [{SKELETON}] nothing packed")
        return pd.DataFrame()
    poses = np.stack(arrs)
    np.save(os.path.join(mout, "poses.npy"), poses)
    df = pd.DataFrame(rows)
    df.to_parquet(os.path.join(mout, "index.parquet"), index=False)
    print(f"  [{SKELETON}] {len(df)} clips, array {poses.shape} "
          f"{poses.nbytes/1e6:.1f} MB, {(time.time()-t0)/60:.1f} min", flush=True)
    return df

## Download

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

REPO = "Kevin-Pal/CUHK-X_Small_Model_Track"
t = time.time()
snapshot_download(repo_id=REPO, repo_type="dataset", local_dir=RAW,
                  allow_patterns=["Small-Model-Track/**"], max_workers=8,
                  token=os.environ["HF_TOKEN"])
print(f"\ndownloaded {tsize(RAW):.1f} GB in {(time.time()-t)/60:.1f} min\n{disk()}")

SRC      = f"{RAW}/Small-Model-Track"
HAR_ZIP  = f"{SRC}/Training/data/HAR.zip"      # last volume of HAR.z01..z08
TEST_ZIP = f"{SRC}/Testing/data/small_model_track_test.zip"
sh(f"ls -la {SRC}/Training/data {SRC}/Testing/data")

## Extract the test set and confirm the layout

In [ ]:
sh(f"7z x '{TEST_ZIP}' -o'{EXTRACT}' -y -bso0 -bsp0")

import prep
train_root, test_root = prep.find_roots(EXTRACT)
print(f"train_root: {train_root}\ntest_root : {test_root}")

clips = sorted(d for d in os.listdir(test_root) if d.startswith("SM_test"))
print(f"test clips: {len(clips)}   (expected 405)")

from collections import Counter
pres = Counter()
for c in clips:
    for m in os.listdir(os.path.join(test_root, c)):
        if os.path.isdir(os.path.join(test_root, c, m)):
            pres[m] += 1
for m, n in pres.most_common():
    print(f"  {m:<14} {n:>4}/{len(clips)}  ({100*n/len(clips):5.1f}%)")

## Extract → pack → delete, one modality at a time

Selective extraction (`-ir!`) is what keeps peak disk inside Kaggle's budget:
the archives stay on disk but only one modality is ever unpacked at a time.

In [ ]:
WORKERS = os.cpu_count() or 4
summary = {}

for mod in MODALITIES:
    print(f"\n{'='*70}\n{mod}\n{'='*70}", flush=True)
    mdir = f"{EXTRACT}/HAR/data/{mod}"

    if not os.path.isdir(mdir):
        t = time.time()
        sh(f"7z x '{HAR_ZIP}' -o'{EXTRACT}' -y -bso0 -bsp0 -ir'!HAR/data/{mod}/*'",
           check=False)
        print(f"  extracted in {(time.time()-t)/60:.1f} min  |  {disk()}")
    if not os.path.isdir(mdir):
        print(f"  !! {mdir} missing — inspect the archive layout:")
        sh(f"7z l '{HAR_ZIP}' -ba | head -20", check=False)
        continue

    train_root, test_root = prep.find_roots(EXTRACT)
    recs = prep.discover_train(train_root, mod) + prep.discover_test(test_root, mod)
    ntr = sum(r["split"] == "train" for r in recs)
    print(f"  clips: {len(recs)}  ({ntr} train / {len(recs)-ntr} test)", flush=True)
    if not recs:
        continue

    if mod == "Skeleton":
        df = prep.pack_skeleton(recs, OUT, SKEL_FRAMES, WORKERS)
    else:
        df = prep.pack_images(mod, recs, OUT, FRAMES, SIZE, CROP, QUALITY, WORKERS)

    if len(df):
        trn = df[df.split == "train"]
        summary[mod] = dict(clips=int(len(df)), train=int(len(trn)),
                            test=int((df.split == "test").sum()),
                            users=sorted(trn.user.astype(str).unique()),
                            classes=int(trn.action_id.nunique()))
        print(f"  users: {summary[mod]['users']}")
        print(f"  classes covered: {summary[mod]['classes']}/40")

    shutil.rmtree(mdir, ignore_errors=True)     # reclaim before the next modality
    print(f"  freed {mod}  |  {disk()}", flush=True)

## Ship the label map and submission template

In [ ]:
for src, dst in [(f"{SRC}/class_mapping.csv", "class_mapping.csv"),
                 (f"{SRC}/Testing/test_file/test.csv", "test.csv"),
                 (f"{SRC}/Testing/test_file/sample_submission.csv",
                  "sample_submission.csv")]:
    if os.path.exists(src):
        shutil.copy2(src, f"{OUT}/{dst}")

with open(f"{OUT}/prep_config.json", "w") as fh:
    json.dump(dict(modalities=MODALITIES, frames=FRAMES, skel_frames=SKEL_FRAMES,
                   size=SIZE, quality=QUALITY, crop=CROP, summary=summary),
              fh, indent=2)

print(json.dumps(summary, indent=2)[:2500])
print(f"\ncompact output: {tsize(OUT):.2f} GB")
print(f"total runtime : {(time.time()-T0)/60:.1f} min")
sh(f"du -sh {OUT}/* | sort -h", check=False)

## Check before moving on

* **train users must be 1–9 and 16–24**, and none of 10, 11, 25, 26 —
  the test subjects. If a test user shows up in training, the split is wrong.
* every modality should cover **40/40 classes**
* test counts should be 405 (395 for Thermal, which is genuinely missing on
  10 clips)

Then `Save Version → Save & Run All (Commit)`; attach the output as the input
dataset for Step 2.